# 04 · Feature Engineering

Compute targets, bin severity, assign temporal split, impute missing, and export the Analytic Base Table (ABT) as CSV + XLSX.

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
from src.feature_engineering import (
    compute_targets, bin_severity, assign_split, impute_missing,
)
df = pd.read_csv(INTERIM / 'fused_zip_hurricane.csv', dtype={'zip_code': str})
df['zip_code'] = df['zip_code'].str.zfill(5)
df = assign_split(df)
df = compute_targets(df)
df['damage_severity_class'] = bin_severity(df[TARGET_COL])
df = impute_missing(df)
print(df.shape); df['damage_severity_class'].value_counts()

(30161, 65)


damage_severity_class
Low       17907
Severe     4419
High       3477
Medium     3451
Name: count, dtype: int64

In [3]:
# Class distribution per split
df.groupby(['train_test_split','damage_severity_class']).size().unstack(fill_value=0)

damage_severity_class,Low,Medium,High,Severe
train_test_split,,,,
TEST,12799,2314,1845,2103
TRAIN,3836,798,1207,1932
VAL,1272,339,425,384


In [4]:
# Save ABT
abt_csv = PROC / 'abt.csv'; df.to_csv(abt_csv, index=False)
print('wrote', abt_csv, df.shape)

wrote C:\Users\chaitanya\Documents\ML Project\data\processed\abt.csv (30161, 65)


### Export XLSX with 3 sheets

In [5]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font
group_colors = {
    'identifiers':'BDD7EE','demographics':'C6E0B4','svi':'FFE699',
    'food_access':'F4B084','flood':'9DC3E6','storm':'D9D9D9',
    'targets':'F8CBAD','derived_output':'E4BCF6'}
col_group = {c:g for g, cols in FEATURE_GROUPS.items() for c in cols}
wb = Workbook()
ws = wb.active; ws.title = 'ABT'
ws.append(list(df.columns))
for i, c in enumerate(df.columns, 1):
    g = col_group.get(c)
    if g:
        ws.cell(row=1, column=i).fill = PatternFill('solid', fgColor=group_colors.get(g, 'FFFFFF'))
        ws.cell(row=1, column=i).font = Font(bold=True)
for row in df.itertuples(index=False):
    ws.append(list(row))
dd = wb.create_sheet('Data Dictionary')
dd.append(['column','group','dtype','description'])
for c in df.columns:
    dd.append([c, col_group.get(c,'other'), str(df[c].dtype), ''])
ss = wb.create_sheet('Summary Stats')
ss.append(['n_rows', len(df)])
ss.append(['n_zips', df['zip_code'].nunique()])
ss.append(['n_hurricanes', df['disaster_number'].nunique()])
for split, n in df['train_test_split'].value_counts().items():
    ss.append([f'split_{split}', int(n)])
wb.save(str(PROC / 'abt.xlsx')); print('saved abt.xlsx')

saved abt.xlsx
